# FORS2-SPEC Spectra Inspection — V1369 Cen

Goal: download two FORS2-SPEC FITS files from the ESO archive for V1369 Cen
and inspect every HDU to understand why these spectra fail validation via
our `eso_fallback` profile.

The fallback profile expects:
- `HDU[0]`: PrimaryHDU with NAXIS=0, metadata (INSTRUME, MJD-OBS, etc.)
- `HDU[1]`: BinTableHDU with columns WAVE, FLUX (and optionally ERR, QUAL, SNR)

We want to see what FORS2 *actually* delivers.

In [ ]:
# Imports
import os
import tempfile
from getpass import getpass
from pathlib import Path

import numpy as np
import pyvo as vo
from astropy.coordinates import SkyCoord
from astropy.io import fits
from astropy.units import Quantity
from astroquery.eso import Eso
import requests

## 1. Query ESO SSAP for FORS2 spectra of V1369 Cen

In [ ]:
# V1369 Cen coordinates (ICRS)
coord = SkyCoord.from_name("V1369 Cen")
print(f"Resolved coordinates: RA={coord.ra.deg:.6f}, Dec={coord.dec.deg:.6f}")

In [ ]:
# SSAP cone search — same endpoint our discoverer uses
SSAP_ENDPOINT = "http://archive.eso.org/ssap"
SEARCH_DIAMETER_DEG = 20 / 3600  # 20 arcsec

service = vo.dal.SSAService(SSAP_ENDPOINT)
results = service.search(
    pos=coord,
    diameter=Quantity(SEARCH_DIAMETER_DEG, "deg"),
)

print(f"Total results: {len(results)}")

# Show available columns
print("\nColumn names:")
print(list(results.fieldnames))

In [ ]:
# Filter to FORS2 results only
fors2_rows = [
    row for row in results
    if "FORS2" in str(row.get("COLLECTION", "")).upper()
]
print(f"FORS2 results: {len(fors2_rows)}")

for i, row in enumerate(fors2_rows):
    print(f"\n--- FORS2 spectrum {i} ---")
    for field in ["COLLECTION", "TARGETNAME", "CREATORDID", "access_url",
                  "em_min", "em_max", "SPECRP", "SNR", "t_min", "t_max"]:
        val = row.get(field, "N/A")
        print(f"  {field:20s}: {val}")

## 2. Download two FORS2 spectra

In [ ]:
# Pick the first two FORS2 spectra
assert len(fors2_rows) >= 2, f"Need at least 2 FORS2 spectra, found {len(fors2_rows)}"

download_dir = Path(tempfile.mkdtemp(prefix="fors2_inspect_"))

# ESO dataportal requires authentication — use astroquery.eso
eso = Eso()
eso.login()  # will prompt for credentials (or use stored keyring)

fits_paths = []
for i, row in enumerate(fors2_rows[:2]):
    # The access_url contains the ADP dataset identifier
    url = str(row["access_url"])
    # Extract the ADP id from the URL (e.g. 'ADP.2025-07-14T10:25:56.794')
    adp_id = url.rsplit("/", 1)[-1]
    print(f"Spectrum {i}: ADP ID = {adp_id}")
    print(f"  access_url = {url}")

    # astroquery.eso.retrieve_data downloads to cache and returns paths
    downloaded = eso.retrieve_data([adp_id], destination=str(download_dir))
    # Find the .fits file(s) returned
    for p in downloaded:
        p = Path(p)
        # astroquery may return .fits.Z or .fits — handle both
        if p.suffix == '.Z':
            import subprocess
            subprocess.run(['uncompress', str(p)], check=True)
            p = p.with_suffix('')
        print(f"  → {p.name} ({p.stat().st_size:,} bytes)")
        fits_paths.append(p)

print(f"\nDownloaded {len(fits_paths)} files to {download_dir}")

## 3. Inspect HDU structure of each file

In [ ]:
for path in fits_paths:
    print("=" * 80)
    print(f"FILE: {path.name}")
    print("=" * 80)

    with fits.open(path) as hdulist:
        hdulist.info()
        print()

        for idx, hdu in enumerate(hdulist):
            print(f"--- HDU[{idx}]: {type(hdu).__name__} (name={hdu.name!r}) ---")
            print(f"  NAXIS  = {hdu.header.get('NAXIS', 'N/A')}")
            for k in range(1, 5):
                nk = f"NAXIS{k}"
                if nk in hdu.header:
                    print(f"  {nk} = {hdu.header[nk]}")

            # If it's a BinTableHDU, show column info
            if hasattr(hdu, 'columns') and hdu.columns is not None:
                print(f"\n  BinTable columns ({len(hdu.columns)}):")
                for col in hdu.columns:
                    print(f"    {col.name:20s}  format={col.format!r:10s}  unit={col.unit!r}")
                # Show shape of data
                if hdu.data is not None:
                    print(f"\n  Data shape: {hdu.data.shape}  (nrows={len(hdu.data)})")
                    # For each column, show shape and first few values
                    for col in hdu.columns:
                        arr = hdu.data[col.name]
                        print(f"    {col.name}: shape={np.shape(arr)}, dtype={arr.dtype}")
                        flat = np.asarray(arr).flatten()
                        if len(flat) > 0:
                            print(f"      first 5: {flat[:5]}")
                            print(f"      last  5: {flat[-5:]}")
                            finite = flat[np.isfinite(flat)] if np.issubdtype(flat.dtype, np.floating) else flat
                            if len(finite) > 0:
                                print(f"      min={np.min(finite)}, max={np.max(finite)}, len={len(flat)}")

            # If it's an ImageHDU with data, show shape
            elif hdu.data is not None:
                print(f"  Image data shape: {hdu.data.shape}, dtype={hdu.data.dtype}")
                flat = hdu.data.flatten()
                finite = flat[np.isfinite(flat)] if np.issubdtype(flat.dtype, np.floating) else flat
                if len(finite) > 0:
                    print(f"  min={np.min(finite):.6g}, max={np.max(finite):.6g}")

            print()

## 4. Dump key header keywords

Show the keywords our `eso_fallback` profile looks for.

In [ ]:
KEYWORDS_OF_INTEREST = [
    "INSTRUME", "TELESCOP", "OBJECT", "RA", "DEC",
    "MJD-OBS", "DATE-OBS", "EXPTIME",
    "SPEC_RES", "SPECRP",
    "NAXIS", "NAXIS1", "NAXIS2",
    "CRVAL1", "CDELT1", "CRPIX1", "CTYPE1", "CUNIT1",
    "CRVAL2", "CDELT2", "CRPIX2", "CTYPE2", "CUNIT2",
    "CD1_1", "CD2_2",
    "DISPELEM", "FILTER1",
    "PRODCATG", "FLUXCAL",
]

for path in fits_paths:
    print("=" * 80)
    print(f"FILE: {path.name}")
    print("=" * 80)

    with fits.open(path) as hdulist:
        for idx, hdu in enumerate(hdulist):
            print(f"\n--- HDU[{idx}] header keywords of interest ---")
            for kw in KEYWORDS_OF_INTEREST:
                if kw in hdu.header:
                    print(f"  {kw:15s} = {hdu.header[kw]!r}")

            # Also show ALL TTYPE/TFORM/TUNIT keywords (column metadata)
            for key in sorted(hdu.header.keys()):
                if key.startswith(("TTYPE", "TFORM", "TUNIT")):
                    print(f"  {key:15s} = {hdu.header[key]!r}")
    print()

## 5. Full header dump (for reference)

Uncomment to see complete headers if the above isn't enough.

In [ ]:
# for path in fits_paths:
#     with fits.open(path) as hdulist:
#         for idx, hdu in enumerate(hdulist):
#             print(f"\n{'=' * 80}")
#             print(f"{path.name} — HDU[{idx}] FULL HEADER")
#             print(f"{'=' * 80}")
#             print(repr(hdu.header))